# Pandas Workshop

## Workshop goal

By the end, you should be able to use Pandas to:

- refresh core operations quickly: loading, inspecting, filtering, sorting, missing values, and basic stats
- think in vectorized column operations instead of loops
- create useful features from names, family counts, tickets, cabins, and fares
- use advanced `groupby`, `transform`, ranking, and `pivot_table`
- create useful visualizations

If you have any questions feel free to message me at ines@artaicare.com.

# 0. Setup

We will load the Titanic dataset from a public CSV.


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic_raw = pd.read_csv(url)

titanic_raw.head()

## Column guide

| Column | Meaning |
|---|---|
| `PassengerId` | Unique passenger identifier |
| `Survived` | Target: `0` = did not survive, `1` = survived |
| `Pclass` | Passenger class: `1` = first, `2` = second, `3` = third |
| `Name` | Passenger name |
| `Sex` | Passenger sex |
| `Age` | Passenger age |
| `SibSp` | Number of siblings/spouses aboard |
| `Parch` | Number of parents/children aboard |
| `Ticket` | Ticket number |
| `Fare` | Ticket fare |
| `Cabin` | Cabin number; many values are missing |
| `Embarked` | Port of embarkation: `C`, `Q`, `S` |


# 1. Pandas refresher

Work in pairs. Try to finish quickly without searching online.

## Challenge 1A: inspect the dataset

Answer with code:

1. How many rows and columns are there?
2. What are the data types?
3. Which columns have missing values?
4. What are the first 5 rows?
5. What are the unique values in `Pclass`, `Sex`, and `Embarked`?

In [ ]:
## add your code here

## Challenge 1B: select, filter, sort

Create a DataFrame containing passengers who:

- were female
- were younger than 18
- survived
- are sorted by `Fare` from highest to lowest

Show only:

`Name`, `Sex`, `Age`, `Pclass`, `Fare`, `Survived`

In [ ]:
## add your code here

## Challenge 1C: quick survival questions

1. What percentage of passengers survived?
2. What was the survival rate by sex?
3. What was the survival rate by passenger class?
4. Who paid the highest fare?
5. What is the average age of survivors vs non-survivors?

In [ ]:
## add your code here

## Challenge 1D: make a clean working copy

Create a working DataFrame named `df`.

Requirements:

- do not modify `titanic_raw` directly
- standardize column names to lowercase
- fill missing `embarked` values with the most common value
- fill missing `age` with the median age
- create a column called `cabin_known` that is `True` if cabin is not missing

In [ ]:
## add your code here

# 2. Vectorized thinking

Pandas is powerful when you operate on entire columns.

Avoid this mindset:

```python
for i in range(len(df)):
    ...
```

Prefer this mindset:

```python
df["new_column"] = df["old_column"] > value
```

## From loop thinking to column thinking

We will create a passenger age group.

In [ ]:
## loop thinking: it works but its not the most efficient

age_labels = []
for age in df["age"]:
    if age < 16:
        age_labels.append("child")
    elif age < 60:
        age_labels.append("adult")
    else:
        age_labels.append("senior")

df["age_group_loop"] = age_labels

df[["age", "age_group_loop"]].head(10)

In [ ]:
## lets drop the previous created column
df = df.drop("age_group_loop", axis=1)

## vectorized
df["age_group"] = "adult"

df.loc[df["age"] < 16, "age_group"] = "child"
df.loc[df["age"] >= 60, "age_group"] = "senior"


df[["age", "age_group"]].head(10)

## Challenge 2A: create columns without loops

Create these columns:

1. `is_child`: passenger is younger than 16
2. `is_first_class`: passenger is in first class
3. `paid_high_fare`: fare is above the median fare
4. `port_name`: map `C`, `Q`, `S` to full port names
5. `fare_level`: map `free_or_low` (0-10), `medium` (10-30), `high`(30-100), or `very_high`(100-80000)


In [ ]:
## add your code here

# 3. Feature engineering

A **feature** is a column that may help explain or predict an outcome.

Today's outcome is:

> Did the passenger survive?


## Extract titles from names

Names contain social titles such as `Mr`, `Mrs`, `Miss`, `Master`, `Dr`, and more.

In [ ]:
## first look at the name column
df['name']

In [ ]:
df["title"] = df["name"].str.extract(r",\s*([^.]*)\.")

df["title"].value_counts()


Many titles are rare. For analysis and ML prep, we can group them.

In [ ]:
common_titles = ["Mr", "Mrs", "Miss", "Master"]
df["title_grouped"] = df["title"].where(df["title"].isin(common_titles), "Rare")

df["title_grouped"].value_counts()

## Challenge 3A: build Titanic features

Create some features that might help explain survival.

Possible ideas:

- `family_size`
- `is_alone`
- `deck`, extracted from the first letter of cabin
- `fare_per_person`
- `large_family`


In [ ]:
## add your code here

# 4. GroupBy investigation

Now use Pandas to investigate historical questions.

Basic `groupby` answers a question like:

> What is the survival rate for each group?

In [ ]:
sex_summary = df.groupby("sex").agg(
    passengers=("passengerid", "count"),
    survival_rate=("survived", "mean"),
    avg_age=("age", "mean"),
    avg_fare=("fare", "mean"),
)

sex_summary

## `transform`: group information back onto each passenger

Use `transform` when you want to compare each passenger to their group.

In [ ]:
df["class_avg_fare"] = df.groupby("pclass")["fare"].transform("mean")
df["fare_vs_class_avg"] = df["fare"] - df["class_avg_fare"]

df[["name", "pclass", "fare", "class_avg_fare", "fare_vs_class_avg"]].head(10)

## Challenge 4A: investigation

Choose some questions to investigate:

1. Did women survive more often than men?
2. Which passenger class had the highest survival rate?
3. Did children survive more often than adults?
4. Did passengers traveling alone survive more or less often?
5. Which title group had the highest survival rate?
6. Which deck had the highest survival rate? Is this result trustworthy?
7. Did passengers who paid more than their class average survive more often?
8. Who was the highest-fare passenger in each class?
9. Rank passengers by fare within each class.
10. Which family size had the best survival rate?

In [ ]:
## add your code here

# 5. Pivot tables

> How did survival depend on both passenger class and sex?

In [ ]:
survival_pivot = df.pivot_table(
    values="survived",
    index="pclass",
    columns="sex",
    aggfunc="mean"
)

survival_pivot


A survival rate can be misleading if the group is tiny. Always check counts.

In [ ]:
count_pivot = df.pivot_table(
    values="survived",
    index="pclass",
    columns="sex",
    aggfunc="count"
)

count_pivot

## Challenge 5A: build your own pivot tables

1. Survival rate by `title_grouped` and `pclass`
2. Average fare by `port_name` and `pclass`
3. Survival rate by `age_group` and `sex`
4. Count of passengers by `deck` and `pclass`


In [ ]:
## add your code here

# 6. Data investigations


## Challenge 6A:


1. **Did social status matter?** Find evidence using passenger class, fare, or deck.
2. **Was “women and children first” visible in the data?** Compare men, women, boys, and girls.
3. **Did passengers travelling alone have worse outcomes?** Use `is_alone` and survival.
4. **Did families help or hurt survival?** Investigate `family_size`.
5. **Did paying more always mean better survival?** Compare fare, fare level, and class.
6. **Who paid unusually high fares for their class?** Use `fare_vs_class_avg`.
7. **Were first-class passengers always the highest-paying passengers?** Investigate fare by class.
8. **Do titles reveal anything about survival?** Use `title_grouped` or `title`.
9. **Do cabin decks reveal anything useful?** Use `deck`, but check whether the result is trustworthy.


In [ ]:
##add your code

# 7. Visualizations

Now we will turn some of those results into simple visuals.


## Challenge 7A: Figures

Choose 2 or 3 charts from the list below:

1. Survival rate by sex
2. Survival rate by passenger class
3. Survival rate by sex and passenger class using a pivot table
4. Age distribution
5. Fare distribution
6. Survival rate by fare level
7. Survival rate by family size
8. Passenger count by deck


In [ ]:
## add your code here
